# htd eval arms

Generates transcripts for `sft_s1` and `sft_v5` only (same generation code as `kaggle/eval`), with adapter sha256 sidecars, so eval v2 reuses them.


In [ ]:
import json, os, shutil, subprocess, sys, importlib.util
from pathlib import Path

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)

def find_repo():
    hits = [p for p in Path("/kaggle/input").rglob("rule_checker.py")
            if p.exists() and (p.parents[2] / "scripts" / "train.py").exists()]
    exact = [p for p in hits if "datasets/rockerleo/htd-code" in str(p)]
    hits = exact or hits
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit("code dataset not mounted")
    return hits[0].parents[2]

WORK = Path("/kaggle/working/htd")
if WORK.exists():
    shutil.rmtree(WORK)
SRC = find_repo()
shutil.copytree(SRC, WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))
print("code root:", SRC)

MARKERS = {
    "scripts/train_aligntune.py": "--method",
    "src/htd/audit.py": "auditkit not installed",
    "src/htd/rule_checker.py": "script_ok",
    "data/product_sheet.json": "प्रीमेच्योर",
    "kaggle/eval/eval.ipynb": "sft_v5",
    "kaggle/eval_arms/eval_arms.ipynb": "ARMS = ",
}
for rel, marker in MARKERS.items():
    text = (WORK / rel).read_text(encoding="utf-8")
    assert marker in text, f"stale code dataset: {rel} lacks {marker!r}"
print("code markers ok")

for p in Path("/kaggle/input").rglob("hf_token.txt"):
    os.environ.setdefault("HF_TOKEN", p.read_text().strip())
    os.environ.setdefault("HUGGINGFACE_API_KEY", p.read_text().strip())
assert os.environ.get("HF_TOKEN"), "attach rockerleo/htd-secrets"

# Every dependency is installed before any model load, so a missing package
# cannot waste an hour of generation.
run([sys.executable, "-m", "pip", "install", "-q",
     "curatorkit[generation] @ git+https://github.com/Lexsi-Labs/CuratorKIT.git"])
run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
assert importlib.util.find_spec("torchao") is None, "torchao is still installed"
import curatorkit  # noqa: F401
import litellm  # noqa: F401
print("dependencies ok")

FIRE = "CohereLabs/tiny-aya-fire"
print(run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
for split, n in (("test", 20), ("heldout", 20), ("control_test", 20)):
    run([sys.executable, "scripts/build_pilot.py", "--split", split,
         "--n-per-script", str(n), "--out", f"data/generated/{split}_dialogues.jsonl"])

In [ ]:
hits = sorted(Path("/kaggle/input").rglob("fire_test_plain.jsonl"))
scoped = [p for p in hits if "htd-base-eval" in str(p)]
hits = scoped or hits
assert len(hits) == 1, f"expected one base transcript dir, found {len(hits)}"
base_dir = hits[0].parent
Path("outputs/eval").mkdir(parents=True, exist_ok=True)
# Earlier arm transcripts (base and trained) are reused; an arm is regenerated only if
# its file is missing or its dialogue ids differ from the current split.
for p in [*base_dir.glob("*.jsonl"), *base_dir.glob("*.adapter.json")]:
    shutil.copy(p, f"outputs/eval/{p.name}")
print("reusable transcripts:", sorted(p.name for p in base_dir.glob("*.jsonl")))
print("adapter hashes:", sorted(p.name for p in base_dir.glob("*.adapter.json")))

def ids(path):
    return {json.loads(line)["dialogue_id"] for line in Path(path).read_text().splitlines() if line.strip()}

checks = {
    "test": ["fire_test_plain.jsonl", "fire_test_strict.jsonl", "fire_test_third.jsonl"],
    "heldout": ["fire_heldout_plain.jsonl"],
    "control_test": ["fire_control_plain.jsonl"],
}
for split, names in checks.items():
    built = ids(f"data/generated/{split}_dialogues.jsonl")
    for name in names:
        got = ids(f"outputs/eval/{name}")
        assert got == built, f"base transcripts differ for {split}/{name}: {len(got)} vs {len(built)}"
print("base transcripts reused and id-matched")


In [ ]:
def adapter_path(name):
    hits = [p.parent for p in Path("/kaggle/input").rglob("adapter_config.json") if p.parent.name == name]
    hits.sort(key=lambda p: len(p.parts))
    return str(hits[0]) if hits else None

def dataset_adapter(slug):
    """Adapter files uploaded flat into a dataset: find them by the dataset slug."""
    hits = [p.parent for p in Path("/kaggle/input").rglob("adapter_config.json") if f"/{slug}/" in str(p)]
    hits.sort(key=lambda p: len(p.parts))
    return str(hits[0]) if hits else None

FOUND = {
    "dpo5": adapter_path("dpo_b01_lr5e6"),
    "dpo1": adapter_path("dpo_b01_lr1e5"),
    "sft": adapter_path("sft_lr1e4"),
    "at": adapter_path("aligntune_dpo"),
    "at_sft": adapter_path("aligntune_sft"),
    "sft2": adapter_path("sft2_lr1e4"),
    "dpo_short": adapter_path("dpo_short_lr5e6"),
    "at_drop": adapter_path("aligntune_dpo_drop"),
    "sft_s1": adapter_path("sft_lr1e4_s1"),
    "sft_v5": adapter_path("sft_lr1e4_v5") or dataset_adapter("htd-sft-v5"),
}
# Transcripts only, for the arms listed here; eval v2 reuses them by adapter sha256.
ARMS = {"sft_s1", "sft_v5"}
FOUND = {k: v for k, v in FOUND.items() if v and k in ARMS}
print(json.dumps(FOUND, indent=2))
REQUIRED = ARMS
missing = REQUIRED - set(FOUND)
assert not missing, f"missing required adapters: {sorted(missing)}"
import hashlib

def adapter_sha(path):
    return hashlib.sha256((Path(path) / "adapter_model.safetensors").read_bytes()).hexdigest()

SHA = {label: adapter_sha(path) for label, path in FOUND.items()}
# Adapters whose weights are already known must match exactly (guards against a changed kernel output).
EXPECTED = {"sft": "85162fabffb954a3", "at": "91ebdfcf05cdadc1", "sft_v5": "56cc24690269ab4a"}
for label, prefix in EXPECTED.items():
    if label in SHA:
        assert SHA[label].startswith(prefix), f"{label}: adapter sha {SHA[label][:16]} != expected {prefix}"
print("adapter sha:", {k: v[:12] for k, v in SHA.items()})
assert len(set(SHA.values())) == len(SHA), "two systems share one adapter"
for label, path in FOUND.items():
    assert (Path(path) / "adapter_model.safetensors").stat().st_size > 1_000_000, label
print("required adapters ok")

In [ ]:
SPLIT_FILE = {"test": "test", "heldout": "heldout", "control_test": "control_test"}
SPLIT_TAG = {"test": "test", "heldout": "heldout", "control_test": "control"}
failed = []
done = []
for label, adapter in FOUND.items():
    for split in ("test", "heldout", "control_test"):
        tag = f"{label}_{SPLIT_TAG[split]}"
        reused, sidecar = Path(f"outputs/eval/{tag}.jsonl"), Path(f"outputs/eval/{tag}.adapter.json")
        # Reuse only a transcript made by this exact adapter (sha256 of its weights).
        if (reused.exists() and sidecar.exists()
                and json.loads(sidecar.read_text())["sha256"] == SHA[label]
                and ids(reused) == ids(f"data/generated/{SPLIT_FILE[split]}_dialogues.jsonl")):
            print(f"reusing {tag}", flush=True)
            done.append(tag)
            continue
        reused.unlink(missing_ok=True)
        cmd = [sys.executable, "scripts/run_pilot.py", "--dialogues", f"data/generated/{SPLIT_FILE[split]}_dialogues.jsonl",
               "--model", FIRE, "--adapter", adapter, "--model-name", label, "--variant", "plain",
               "--backend", "hf", "--device", "cuda", "--batch-size", "8", "--max-new-tokens", "384",
               "--out", f"outputs/eval/{tag}.jsonl"]
        try:
            run(cmd)
            assert Path(f"outputs/eval/{tag}.jsonl").exists()
            assert ids(f"outputs/eval/{tag}.jsonl") == ids(f"data/generated/{SPLIT_FILE[split]}_dialogues.jsonl"), tag
            sidecar.write_text(json.dumps({"adapter": adapter, "sha256": SHA[label]}))
            done.append(tag)
        except Exception as e:
            print(f"ARM FAILED: {tag}: {e}", flush=True)
            failed.append(tag)
print("done:", done)
print("failed:", failed)

In [ ]:
import zipfile
ART = Path("/kaggle/working/htd-artifacts")
if ART.exists():
    shutil.rmtree(ART)
ART.mkdir(parents=True)
for tag in done:
    for suffix in (".jsonl", ".adapter.json"):
        shutil.copy(f"outputs/eval/{tag}{suffix}", ART / f"{tag}{suffix}")
print("packaged:", sorted(p.name for p in ART.iterdir()))
print("FAILED:", failed)
assert not failed, f"arms failed: {failed}"
assert sorted(done) == sorted(f"{a}_{s}" for a in ARMS for s in ("test", "heldout", "control")), done